In [ ]:
%pip -q install -r requirements.txt
import os
import shutil
import sys
from pathlib import Path

# Also works when VS Code starts the kernel without sourcing local-env.sh.
os.environ["PATH"] = str(Path(sys.executable).parent) + os.pathsep + os.environ.get("PATH", "")
os.environ.setdefault("XDG_CACHE_HOME", str(Path.cwd() / ".cache"))
os.environ.setdefault("HF_HOME", str(Path.cwd() / ".cache" / "huggingface"))
if not shutil.which("ffmpeg"):
    raise RuntimeError("FFmpeg not found. Run bash setup-local.sh in the project terminal first.")
print("Dependencies and FFmpeg ready.")


Note: you may need to restart the kernel to use updated packages.
Dependencies and FFmpeg ready.


In [7]:
import gc
import re
import time
import torch
import whisper
import pandas as pd
from IPython.display import display

AUDIO_DIR = "./selected_audio"
OUTPUT_CSV = "./model_comparison.csv"
LANGUAGE = "en"
BATCH_STATE = "default"  # Keyword prompt: "decision", "reason", or "default" (both).

# Edit this array to choose models. Each name must be unique (used in CSV columns).
MODELS = [
    {"name": "tiny", "backend": "whisper", "model": "tiny"},
    {"name": "base", "backend": "whisper", "model": "base"},
    {"name": "small", "backend": "whisper", "model": "small"},
    # Full Transformers-format Whisper checkpoint on Hugging Face:
    # {"name": "finetuned", "backend": "huggingface", "model": "norillalearning/YOUR_MODEL_REPO"},
    # If the fine-tuned repo has no tokenizer/processor, add its matching base:
    # {"name": "finetuned", "backend": "huggingface", "model": "norillalearning/YOUR_MODEL_REPO", "processor": "openai/whisper-small"},
]

MODEL_CACHE = Path(".cache").resolve()

def get_model(spec):
    print(f"Loading {spec['name']}: {spec['backend']} / {spec['model']}...")
    if spec["backend"] == "whisper":
        return whisper.load_model(spec["model"], device="cpu", download_root=str(MODEL_CACHE / "whisper"))
    if spec["backend"] == "huggingface":
        from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
        processor = AutoProcessor.from_pretrained(
            spec.get("processor", spec["model"]), cache_dir=str(MODEL_CACHE / "huggingface"),
        )
        model = AutoModelForSpeechSeq2Seq.from_pretrained(
            spec["model"], cache_dir=str(MODEL_CACHE / "huggingface"), dtype=torch.float32,
        )
        if model.config.model_type != "whisper":
            raise ValueError("The huggingface backend expects a Whisper speech-to-text checkpoint.")
        return pipeline(
            "automatic-speech-recognition", model=model,
            tokenizer=processor.tokenizer, feature_extractor=processor.feature_extractor,
            device=-1,
        )
    raise ValueError(f"Unsupported backend: {spec['backend']}")

def transcribe_audio(model, spec, audio, prompt):
    # All models run on CPU/float32 with the same audio, language and keyword prompt.
    if spec["backend"] == "whisper":
        return model.transcribe(
            audio, initial_prompt=prompt, language=LANGUAGE,
            fp16=False, temperature=0.0, beam_size=5,
        )["text"].strip()
    kwargs = {"num_beams": 5, "temperature": 0.0, "do_sample": False}
    if getattr(model.model.generation_config, "is_multilingual", True):
        kwargs.update(language=LANGUAGE, task="transcribe")
    if prompt:
        kwargs["prompt_ids"] = model.tokenizer.get_prompt_ids(prompt, return_tensors="pt")
    return model(
        {"array": audio.copy(), "sampling_rate": 16000}, generate_kwargs=kwargs,
        return_timestamps=len(audio) > 30 * 16000,
    )["text"].strip()

def normalize_text(text):
    text = str(text).lower().strip()
    return re.sub(r"[^\w\s]", "", text)

print("Models configured:", [spec["name"] for spec in MODELS])
print("Input:", Path(AUDIO_DIR).resolve())
print("Output:", Path(OUTPUT_CSV).resolve())


Models configured: ['tiny', 'base', 'small']
Input: /Users/fandika/Projects/whisper-pipeline/selected_audio
Output: /Users/fandika/Projects/whisper-pipeline/model_comparison.csv


In [8]:
INTENT_CONFIG = {
    "decision": {
        "tower.decision.right": [["right"], ["2"], ["two"], ["second"]],
        "tower.decision.left": [["left"], ["1"], ["number", "one"], ["first"]],
        "tower.decision.same": [["both"], ["together"], ["same", "time"], ["one", "and", "two"]]
    },
    "reason": {
        "tower.reason.tall": [["tall"], ["taller"], ["big"], ["bigger"], ["high"], ["higher"]],
        "tower.reason.weight": [["weight"], ["weighs"], ["top"], ["heavy"], ["heavier"]],
        "tower.reason.thinnerbase": [["thin"], ["thinner"], ["small"], ["smaller"], ["base"], ["bottom"]],
        "tower.reason.notsymmetrical": [["symmetry"], ["symmetric"], ["symmetrical"], ["both", "sides"], ["not", "the", "same"], ["asymmetrical"], ["asymmetric"], ["not", "same"]]
    }
}

def determine_intent(transcript, state):
    """Implementation of the 3-step intent matching logic."""
    if not transcript or str(transcript).strip() == "":
        return "default.noaudio"

    # Step 1: Clean text
    clean = normalize_text(transcript)
    words = clean.split()

    # Step 2: Get intents for current state
    state_intents = INTENT_CONFIG.get(state, {})

    # Step 3: Keyword group matching
    for intent_name, groups in state_intents.items():
        for group in groups:
            # Match if EVERY keyword in group is found as a whole word
            if all(any(re.fullmatch(re.escape(k), w) for w in words) for k in group):
                return intent_name

    return "default.unknown" # Or noaudio if you prefer

In [9]:
KEYWORDS = {
    "reason": [
        "tall", "taller", "big", "high", "higher", "weight", "weighs", "top",
        "heavy", "heavier", "thin", "thinner", "small", "smaller",
        "symmetric", "symmetrical", "both sides", "asymmetrical", "asymmetric"
    ],
    "decision": [
        "right", "two", "left", "one", "both", "together", "same time",
    ],
    "default": [
        "tall", "taller", "big", "high", "higher", "weight", "weighs", "top",
        "heavy", "heavier", "thin", "thinner", "small", "smaller",
        "symmetric", "symmetrical", "both sides", "asymmetrical", "asymmetric",
        "right", "two", "left", "one", "both", "together", "same time",
    ],
}


def run_model_comparison(models, output_csv, state):
    if not models:
        raise ValueError("Add at least one entry to MODELS.")
    names = [spec["name"] for spec in models]
    if any(not name.strip() for name in names) or len(set(names)) != len(names):
        raise ValueError("Every model needs a nonempty, unique name.")
    for spec in models:
        if spec["backend"] not in {"whisper", "huggingface"} or not spec["model"]:
            raise ValueError(f"Invalid model configuration: {spec}")
    if state not in KEYWORDS:
        raise ValueError("BATCH_STATE must be decision, reason, or default.")

    folder = Path(AUDIO_DIR)
    if not folder.is_dir():
        raise FileNotFoundError(f"Audio folder not found: {folder.resolve()}")
    files = sorted(p for p in folder.iterdir() if p.is_file() and p.suffix.lower() == ".wav")
    if not files:
        raise ValueError(f"No WAV files found in {folder.resolve()}")

    # Folder contents determine the rows; no input CSV or existing transcript is needed.
    rows, audio_inputs, audio_errors = [], {}, {}
    for path in files:
        row = {"fileName": path.name, "audio duration [s]": None}
        try:
            audio = whisper.load_audio(str(path))  # Decode once at 16 kHz, outside the timer.
            audio_inputs[path.name] = audio
            row["audio duration [s]"] = round(len(audio) / 16000, 3)
        except Exception as exc:
            audio_errors[path.name] = str(exc)
        rows.append(row)
    df = pd.DataFrame(rows)
    fields = ["model", "transcription", "intent [decision]", "intent [reason]", "processing time [ms]", "status", "error"]
    for spec in models:
        for field in fields:
            df[f"{spec['name']} | {field}"] = None
        df[f"{spec['name']} | model"] = f"{spec['backend']}:{spec['model']}"

    output_path = Path(output_csv)
    if output_path.resolve() in {p.resolve() for p in Path('.').glob('*_audio_log.csv')}:
        raise ValueError("Use a separate OUTPUT_CSV, not an original audio log.")
    output_path.parent.mkdir(parents=True, exist_ok=True)
    prompt = "Try to focus specifically on these words as you listen to the audio: " + ", ".join(KEYWORDS[state])
    print(f"Comparing {len(models)} models on {len(files)} WAV files. CPU / float32.")
    print("Processing time excludes download, model loading, audio decoding and intent classification.")

    for spec in models:
        prefix = spec["name"] + " | "
        model = None
        try:
            model = get_model(spec)  # One model in memory at a time; loading is not timed.
            for idx, row in df.iterrows():
                file_name = row["fileName"]
                try:
                    if file_name in audio_errors:
                        raise ValueError(audio_errors[file_name])
                    start = time.perf_counter()
                    text = transcribe_audio(model, spec, audio_inputs[file_name], prompt)
                    elapsed_ms = (time.perf_counter() - start) * 1000
                    df.at[idx, prefix + "transcription"] = text
                    df.at[idx, prefix + "intent [decision]"] = determine_intent(text, "decision")
                    df.at[idx, prefix + "intent [reason]"] = determine_intent(text, "reason")
                    df.at[idx, prefix + "processing time [ms]"] = round(elapsed_ms, 2)
                    df.at[idx, prefix + "status"] = "ok"
                    print(f"{spec['name']} / {file_name}: {text!r} ({elapsed_ms:.0f} ms)")
                except Exception as exc:
                    df.at[idx, prefix + "status"] = "error"
                    df.at[idx, prefix + "error"] = str(exc)
                    print(f"ERROR {spec['name']} / {file_name}: {exc}")
        except Exception as exc:
            df[prefix + "status"] = "load_error"
            df[prefix + "error"] = str(exc)
            print(f"ERROR loading {spec['name']}: {exc}")
        finally:
            del model
            gc.collect()
            df.to_csv(output_path, index=False)  # Preserve other models' results on failure.

    print(f"Saved {len(df)} rows to {output_path.resolve()}")
    display(df)
    return df

# Each run creates a fresh comparison of the current model list and selected WAVs.
comparison_df = run_model_comparison(MODELS, OUTPUT_CSV, BATCH_STATE)


Comparing 3 models on 3 WAV files. CPU / float32.
Processing time excludes download, model loading, audio decoding and intent classification.
Loading tiny: whisper / tiny...
tiny / 2026-6-14--11-38--16.wav: 'right, two, left, all faced, right, two, left, all faced, right, two, left, all faced, right, all faced, right, all faced, right, all faced, right, all faced, all faced, all faced, all faced, all faced, all faced, all faced, all faced, all faced, all faced, all faced, all faced, all faced, all faced, all faced, all faced, all faced, all faced, all faced, all faced, all faced, all faced, all faced, all faced,' (1376 ms)
tiny / 2026-6-14--13-36--3.wav: 'two, two, three, two, three, two, three, two, three, two, three, two, three, two, three, two, three, two, three, two, three, two, three, two, three, two, three, two, three, two, three, two, three, two, three, two, three, two, three, two, three, two, three, two, three, two, three, two, three, two, three, two, three, two, three, two, th

100%|███████████████████████████████████████| 461M/461M [00:09<00:00, 53.1MiB/s]


small / 2026-6-14--11-38--16.wav: 'Try to fall first' (1130 ms)
small / 2026-6-14--13-36--3.wav: 'See you' (721 ms)
small / 2026-6-14--13-40--5.wav: "Number two, we'll call first. Can you turn on the cell? I'll eat you I already turned" (1944 ms)
Saved 3 rows to /Users/fandika/Projects/whisper-pipeline/model_comparison.csv


,fileName,audio duration [s],tiny | model,tiny | transcription,tiny | intent [decision],tiny | intent [reason],tiny | processing time [ms],tiny | status,tiny | error,base | model,...,base | processing time [ms],base | status,base | error,small | model,small | transcription,small | intent [decision],small | intent [reason],small | processing time [ms],small | status,small | error
0,2026-6-14--11-38--16.wav,5.12,whisper:tiny,"right, two, left, all faced, right, two, left,...",tower.decision.right,default.unknown,1375.55,ok,None,whisper:base,...,365.38,ok,None,whisper:small,Try to fall first,tower.decision.left,default.unknown,1129.8,ok,None
1,2026-6-14--13-36--3.wav,4.80,whisper:tiny,"two, two, three, two, three, two, three, two, ...",tower.decision.right,default.unknown,2581.58,ok,None,whisper:base,...,313.21,ok,None,whisper:small,See you,default.unknown,default.unknown,721.21,ok,None
2,2026-6-14--13-40--5.wav,5.12,whisper:tiny,"with the red chill out for a fast skater, so i...",default.unknown,default.unknown,356.3,ok,None,whisper:base,...,827.08,ok,None,whisper:small,"Number two, we'll call first. Can you turn on ...",tower.decision.right,default.unknown,1943.92,ok,None


In [10]:
# Quick timing summary. Raw transcription and both intents are in OUTPUT_CSV.
summary = []
for spec in MODELS:
    prefix = spec["name"] + " | "
    successful = comparison_df[prefix + "status"].eq("ok")
    timings = pd.to_numeric(comparison_df.loc[successful, prefix + "processing time [ms]"])
    summary.append({
        "model": spec["name"],
        "successful files": int(successful.sum()),
        "failed files": int((~successful).sum()),
        "mean processing time [ms]": round(timings.mean(), 2),
    })
display(pd.DataFrame(summary))
print("Timing compares these local runs; it is not a transcription accuracy score.")


,model,successful files,failed files,mean processing time [ms]
0,tiny,3,0,1437.81
1,base,3,0,501.89
2,small,3,0,1264.98


Timing compares these local runs; it is not a transcription accuracy score.
